# [B1] What does the observed funded STTR collaboration network look like?

**Status:** exploratory; non-citable  
**Research question:** [B1 — Funded STTR collaboration network](../../docs/research-questions.md#b1-sttr-collaboration-network)  
**Decision this informs:** whether the funded award portfolio shows enough repeated firm–research-institution structure to justify identity-resolution and longitudinal follow-up work. It does not rank institution types or estimate program effects.  
**Data as of:** pinned by a verified canonical sidecar or validated operator-declared fallback (see input-status cell)  
**Owner:** STTR exploratory workbench

Within one pinned raw SBIR.gov award export, among canonical `sbir-source-v2` awards whose `Program` value matches the shared whitespace-trimmed, case-insensitive STTR predicate, what do first-observed and repeated firm–research-institution dyads, concentration, partner reuse, agency/year variation, and research-institution-name missingness look like?


## Interpretation boundary — read before any result

1. **This is a funded-portfolio description, not a federal census.** It reads one pinned raw export. It does not establish that every agency record, historical revision, or funded STTR relationship is present.
2. **Research-institution identity is unresolved.** The RI endpoint is only a Unicode/case/whitespace-normalized source name. Different strings can be one institution; one string can refer to several institutions, campuses, departments, or labs. Firm identity uses a valid UEI, then a valid DUNS, then an unresolved surface name. These choices can split or merge nodes and dyads in unknown directions.
3. **“First-observed” is not “newly formed.”** It means the first year a surface-name dyad appears in this selected export. Earlier collaboration can be outside the file or under another spelling.
4. **A blank or placeholder (`N/A`) RI field is typed missingness, not evidence of no research partner.** Awards missing either endpoint stay in the canonical-award denominator and stay out of network topology. Source editions and identity-field conflicts are reported separately.
5. **No institution-type, spinout, or subcontract classification is made.** The notebook does not use token heuristics, seed lists, or the gated STTR cascade.
6. **No access, equity, applicant-pool, selection, or causal claim is authorized.** Award records contain funded recipients, not applicants or a comparison group. Concentration is descriptive canonical-award concentration, not market power or program fairness.


## Data contract

- **Population:** raw CSV source rows whose `Program` value matches `STTR` after surrounding-whitespace removal and case normalization, across observed phases. The shared predicate does not infer program from titles or abstracts.
- **Grain:** raw source edition first; then one canonical award per `sbir_award_grain_key` under `sbir-source-v2`; then one edge per firm-node × RI-surface-name dyad. Canonical edition counts, variant editions, and network-field conflicts remain visible.
- **Keys:** tracking numbers are not assumed unique. A firm node uses canonical `normalize_uei`, then canonical `normalize_duns`, after rejecting single-character placeholder identifiers; unresolved records fall back to `NAME:<surface name>`. An RI node is `NAME:<surface name>`.
- **Inputs:** `SBIR_AWARDS_PATH` may point to a raw SBIR.gov CSV; otherwise the only default is `data/raw/sbir/award_data.csv`. When the sibling `.meta.json` sidecar exists, the canonical verifier checks hash, bytes, row count, and schema fingerprint and supplies source dates. Without a sidecar, matching `SBIR_AWARDS_SHA256` and a valid ISO `SBIR_AWARDS_AS_OF` are required. The canonical CSV reader preserves literal text and validates the ordered header and every row width. No download occurs here.
- **Exclusions:** topology excludes a canonical award when the chosen edition has a blank RI name or lacks valid UEI, valid DUNS, and firm surface name. Missing-year awards remain in overall topology and are excluded only from time tables.
- **Assumptions:** only NFKC Unicode normalization, whitespace collapse, and case-folding are used for name keys. Literal strings such as `N/A` remain text in the source frame; for the RI endpoint only, the placeholder tokens `n/a`, `na`, `none`, `null`, `nan`, and `-` (after case-folding) are typed missingness rather than an institution node, and are counted separately in the coverage table. Punctuation, legal suffixes, acronyms, campus names, and department text remain. All-one-character UEI/DUNS values are treated as placeholders after canonical format normalization. No fuzzy matching occurs.
- **Seed:** `20261028`. No stochastic operation currently uses it; it is reserved for any later displayed sample.
- **Outputs:** in-memory exploratory tables only. No artifact is written. If the raw source, pinned snapshot metadata, or required source schema is absent, the notebook refuses numerical analysis and explains why.


In [ ]:
from __future__ import annotations

import os
import re
import sys
import unicodedata
from datetime import date
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sbir_etl.extractors.sbir_award_export import (  # noqa: E402
    SBIR_GOV_SOURCE_COLUMNS,
    SbirGovSourceError,
    load_verified_award_export,
    ordered_columns_sha256,
    read_sbir_gov_csv,
)
from sbir_etl.extractors.sbir_public_awards import normalize_sbir_awards  # noqa: E402
from sbir_etl.identity.sbir_awards import (  # noqa: E402
    SBIR_AWARD_KEY_VERSION,
    sbir_award_grain_key,
)
from sbir_etl.utils.data.file_io import file_sha256  # noqa: E402
from sbir_etl.utils.identifiers import normalize_duns, normalize_uei  # noqa: E402

# Reuse the shared STTR predicate and column resolver from the STTR D1 spine.
from scripts.sttr_spinout_linkage.d1_spine import (  # noqa: E402
    first_col,
    is_sttr,
)

REPO_ROOT


In [ ]:
RANDOM_SEED = 20261028
MIN_AGENCY_YEAR_AWARDS = 10  # display threshold only; calculations retain every cell
DECLARED_AS_OF = os.getenv("SBIR_AWARDS_AS_OF", "").strip()
DECLARED_SHA256 = os.getenv("SBIR_AWARDS_SHA256", "").strip().lower()

source_override = os.getenv("SBIR_AWARDS_PATH", "").strip()
if source_override:
    AWARDS_PATH = Path(source_override).expanduser()
    SOURCE_SELECTION = "SBIR_AWARDS_PATH override"
else:
    AWARDS_PATH = REPO_ROOT / "data" / "raw" / "sbir" / "award_data.csv"
    SOURCE_SELECTION = "canonical raw CSV default"


def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return f"$SBIR_AWARDS_PATH ({path.name})"


def empty_raw_source() -> pd.DataFrame:
    return pd.DataFrame(columns=SBIR_GOV_SOURCE_COLUMNS)


def load_pinned_snapshot(
    path: Path,
) -> tuple[dict[str, object], pd.DataFrame]:
    status: dict[str, object] = {
        "input": "raw SBIR.gov award_data.csv",
        "selection": SOURCE_SELECTION,
        "path": display_path(path),
        "metadata_path": None,
        "exists": path.is_file(),
        "status": "REFUSED",
        "reason": "",
        "as_of": None,
        "as_of_basis": None,
        "bytes": None,
        "sha256": None,
        "row_count": None,
        "column_count": None,
        "ordered_schema_sha256": None,
        "retrieved_at": None,
        "upstream_published_at": None,
        "snapshot_basis": None,
    }
    if not path.is_file():
        status["reason"] = f"source not found: {display_path(path)}"
        return status, empty_raw_source()
    if path.suffix.lower() != ".csv":
        status["reason"] = "input must be the raw SBIR.gov CSV, not a derived artifact"
        return status, empty_raw_source()

    metadata_path = path.with_suffix(".meta.json")
    if metadata_path.is_file():
        try:
            verified = load_verified_award_export(path, metadata_path)
        except (OSError, SbirGovSourceError) as exc:
            status["reason"] = f"canonical sidecar verification failed: {exc}"
            return status, empty_raw_source()
        metadata = verified.metadata
        upstream = metadata.upstream_published_at
        as_of_timestamp = upstream or metadata.retrieved_at
        status.update(
            {
                "metadata_path": display_path(metadata_path),
                "status": "READY",
                "as_of": as_of_timestamp.date().isoformat(),
                "as_of_basis": (
                    "sidecar upstream_published_at"
                    if upstream is not None
                    else "sidecar retrieved_at; upstream date unavailable"
                ),
                "bytes": metadata.size_bytes,
                "sha256": metadata.sha256,
                "row_count": metadata.row_count,
                "column_count": metadata.column_count,
                "ordered_schema_sha256": metadata.ordered_schema_sha256,
                "retrieved_at": metadata.retrieved_at.isoformat(),
                "upstream_published_at": (
                    upstream.isoformat() if upstream is not None else None
                ),
                "snapshot_basis": "verified canonical sidecar",
            }
        )
        return status, verified.frame

    if not DECLARED_SHA256 or not DECLARED_AS_OF:
        status["reason"] = (
            "canonical sidecar absent; set matching SBIR_AWARDS_SHA256 "
            "and ISO SBIR_AWARDS_AS_OF"
        )
        return status, empty_raw_source()
    if re.fullmatch(r"[0-9a-f]{64}", DECLARED_SHA256) is None:
        status["reason"] = "SBIR_AWARDS_SHA256 must contain 64 hexadecimal characters"
        return status, empty_raw_source()
    try:
        if re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", DECLARED_AS_OF) is None:
            raise ValueError
        declared_as_of = date.fromisoformat(DECLARED_AS_OF).isoformat()
    except ValueError:
        status["reason"] = "SBIR_AWARDS_AS_OF must be a valid YYYY-MM-DD date"
        return status, empty_raw_source()

    try:
        observed_bytes = path.stat().st_size
        observed_sha = file_sha256(path)
    except OSError as exc:
        status["reason"] = f"source inspection failed: {exc}"
        return status, empty_raw_source()
    status.update({"bytes": observed_bytes, "sha256": observed_sha})
    if DECLARED_SHA256 != observed_sha:
        status["reason"] = "observed SHA-256 does not match SBIR_AWARDS_SHA256"
        return status, empty_raw_source()
    try:
        raw = read_sbir_gov_csv(path)
    except (OSError, SbirGovSourceError) as exc:
        status["reason"] = f"canonical CSV validation failed: {exc}"
        return status, empty_raw_source()
    status.update(
        {
            "status": "READY",
            "as_of": declared_as_of,
            "as_of_basis": "operator-declared ISO date",
            "row_count": len(raw),
            "column_count": len(raw.columns),
            "ordered_schema_sha256": ordered_columns_sha256(list(raw.columns)),
            "snapshot_basis": "operator-pinned fallback; canonical sidecar absent",
        }
    )
    return status, raw


snapshot_status, raw_source = load_pinned_snapshot(AWARDS_PATH)
input_status = pd.DataFrame([snapshot_status])
input_status


## Load the STTR source rows

The canonical raw-export reader preserves every CSV field as text and validates the ordered 42-column header and every row width before constructing the frame. A canonical sibling sidecar is verified when present; validated hash/date environment pins are the fallback. The shared STTR predicate trims surrounding whitespace and compares case-insensitively. The full official schema reaches `sbir_award_grain_key`; no key field is projected away first. `normalize_sbir_awards` then selects one deterministic representative edition per canonical award key. RI organization text is joined back by source row; contact names, emails, and phone numbers never enter the analytical view.


In [ ]:
def empty_source_frame() -> pd.DataFrame:
    return pd.DataFrame(
        columns=(*SBIR_GOV_SOURCE_COLUMNS, "source_row", "award_key", "award_key_version")
    )


def empty_portfolio_frame() -> pd.DataFrame:
    return pd.DataFrame(
        columns=(
            *SBIR_GOV_SOURCE_COLUMNS,
            "source_row",
            "award_key",
            "award_key_version",
            "canonical_award_year",
            "source_edition_count",
            "source_edition_variants",
        )
    )


def refused_load(reason: str) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, object]]:
    return empty_source_frame(), empty_portfolio_frame(), {
        "status": "REFUSED",
        "reason": reason,
        "raw_source_rows_scanned": 0,
        "sttr_source_editions": 0,
        "canonical_sttr_awards": 0,
        "edition_rows_collapsed": 0,
    }


def load_sttr_awards(
    raw: pd.DataFrame,
    snapshot: dict[str, object],
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, object]]:
    if snapshot["status"] != "READY":
        return refused_load(str(snapshot["reason"]))

    if tuple(raw.columns) != SBIR_GOV_SOURCE_COLUMNS:
        return refused_load(
            "raw CSV header does not match the canonical 42-column SBIR.gov schema"
        )
    raw["source_row"] = pd.Series(
        range(2, 2 + len(raw)), index=raw.index, dtype="Int64"
    )
    program_col = first_col(raw, ("program", "Program"))
    if program_col is None:
        return refused_load("raw CSV lacks the Program column")
    sttr = raw.loc[raw[program_col].map(is_sttr)].copy()
    if sttr.empty:
        return refused_load("no rows matched the normalized STTR program predicate")

    # Compute identity on full source rows before any analytical projection.
    sttr["award_key"] = sttr.apply(sbir_award_grain_key, axis=1)
    sttr["award_key_version"] = SBIR_AWARD_KEY_VERSION
    normalized = normalize_sbir_awards(sttr)
    chosen = normalized.loc[
        :,
        [
            "award_key",
            "source_row",
            "award_year",
            "source_edition_count",
            "source_edition_variants",
        ],
    ].rename(columns={"award_year": "canonical_award_year"})
    portfolio_raw = sttr.merge(
        chosen, on=["award_key", "source_row"], how="inner", validate="one_to_one"
    )
    report = {
        "status": "READY",
        "reason": "",
        "raw_source_rows_scanned": len(raw),
        "sttr_source_editions": len(sttr),
        "canonical_sttr_awards": len(portfolio_raw),
        "edition_rows_collapsed": len(sttr) - len(portfolio_raw),
    }
    return sttr.reset_index(drop=True), portfolio_raw.reset_index(drop=True), report


In [ ]:
sttr_source, portfolio_raw, load_report = load_sttr_awards(
    raw_source,
    snapshot_status,
)
del raw_source
analysis_ready = (
    load_report["status"] == "READY"
    and not sttr_source.empty
    and not portfolio_raw.empty
)
if not analysis_ready:
    load_report["status"] = "REFUSED"
    if not load_report["reason"]:
        load_report["reason"] = "no STTR rows available for analysis"
pd.DataFrame([load_report])


## Build minimally normalized network endpoints

The original organization strings remain in `portfolio`. Canonical award grain and edition selection come from `sbir-source-v2`; tracking numbers alone are never treated as unique awards. Surface names only standardize Unicode presentation, whitespace, and case. Valid UEI takes precedence over valid DUNS, and unresolved firm name is the final fallback.


In [ ]:
def source_text(value: object) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value)


def display_text(value: object) -> str:
    normalized = unicodedata.normalize("NFKC", source_text(value))
    return " ".join(normalized.split())


def surface_key(value: object) -> str:
    return display_text(value).casefold()


# Literal placeholder strings in `RI Name` are typed missingness, not an institution node.
RI_NULL_TOKENS = frozenset({"n/a", "na", "none", "null", "nan", "-"})


def placeholder_identifier(value: str | None) -> bool:
    return bool(value) and len(set(value)) == 1


def valid_uei(value: object) -> str | None:
    normalized = normalize_uei(value)
    return None if placeholder_identifier(normalized) else normalized


def valid_duns(value: object) -> str | None:
    normalized = normalize_duns(value)
    return None if placeholder_identifier(normalized) else normalized


RAW_TO_VIEW = {
    "Company": "company_name",
    "RI Name": "ri_name",
    "Agency": "agency",
    "Phase": "phase",
    "Agency Tracking Number": "award_id",
    "Contract": "contract",
    "Award Title": "award_title",
    "Proposal Award Date": "award_date",
    "Award Year": "award_year",
    "UEI": "uei",
    "Duns": "duns",
}
VIEW_COLUMNS = [
    *RAW_TO_VIEW.values(),
    "source_row",
    "award_key",
    "award_key_version",
    "canonical_award_year",
    "source_edition_count",
    "source_edition_variants",
]
portfolio = portfolio_raw.rename(columns=RAW_TO_VIEW).loc[:, VIEW_COLUMNS].copy()
portfolio["source_record_id"] = "csv:" + portfolio["source_row"].astype(str)
portfolio["company_name_key"] = portfolio["company_name"].map(surface_key)
portfolio["ri_name_key"] = portfolio["ri_name"].map(surface_key)
portfolio["ri_name_null_token"] = portfolio["ri_name_key"].isin(RI_NULL_TOKENS)
portfolio["uei_key"] = portfolio["uei"].map(valid_uei).fillna("")
portfolio["duns_key"] = portfolio["duns"].map(valid_duns).fillna("")
portfolio["agency_key"] = portfolio["agency"].map(
    lambda value: surface_key(value) or "(missing agency)"
)
portfolio["phase_key"] = portfolio["phase"].map(
    lambda value: surface_key(value) or "(missing phase)"
)


def firm_identity(
    uei_key: str,
    duns_key: str,
    company_name_key: str,
) -> tuple[str, str]:
    if uei_key:
        return f"UEI:{uei_key}", "valid UEI"
    if duns_key:
        return f"DUNS:{duns_key}", "valid DUNS fallback"
    if company_name_key:
        return f"NAME:{company_name_key}", "unresolved surface-name fallback"
    return "", "missing"


firm_identity_values = [
    firm_identity(uei_key, duns_key, company_name_key)
    for uei_key, duns_key, company_name_key in zip(
        portfolio["uei_key"],
        portfolio["duns_key"],
        portfolio["company_name_key"],
        strict=True,
    )
]
portfolio[["firm_key", "firm_identity_basis"]] = pd.DataFrame(
    firm_identity_values,
    index=portfolio.index,
    columns=["firm_key", "firm_identity_basis"],
)
portfolio["ri_key"] = (
    portfolio["ri_name_key"]
    .where(~portfolio["ri_name_null_token"], "")
    .map(lambda value: f"NAME:{value}" if value else "")
)

portfolio["observation_year"] = pd.to_numeric(
    portfolio["canonical_award_year"],
    errors="coerce",
).astype("Int64")
portfolio["network_eligible"] = (
    portfolio["firm_key"].ne("") & portfolio["ri_key"].ne("")
)


def variant_count(series: pd.Series, normalizer) -> int:
    return len({normalizer(value) or "" for value in series})


edition_audit = (
    sttr_source.groupby("award_key", as_index=False)
    .agg(
        observed_source_editions=("source_row", "size"),
        ri_name_variants=("RI Name", lambda s: variant_count(s, surface_key)),
        company_name_variants=("Company", lambda s: variant_count(s, surface_key)),
        valid_uei_variants=("UEI", lambda s: variant_count(s, valid_uei)),
        valid_duns_variants=("Duns", lambda s: variant_count(s, valid_duns)),
        title_variants=("Award Title", lambda s: variant_count(s, surface_key)),
        date_text_variants=("Proposal Award Date", lambda s: variant_count(s, display_text)),
    )
)
edition_audit["network_field_conflict"] = edition_audit[
    ["ri_name_variants", "company_name_variants", "valid_uei_variants", "valid_duns_variants"]
].max(axis=1).gt(1)
analysis_ready = analysis_ready and not portfolio.empty


## Coverage and missingness

Coverage uses canonical `sbir-source-v2` awards as the portfolio denominator. Raw STTR source editions, collapsed edition rows, variant editions, and network-field conflicts remain visible, so topology cannot silently redefine source revisions or missing fields as relationships.


In [ ]:
if analysis_ready:
    missing_ri = portfolio["ri_key"].eq("")
    missing_firm = portfolio["firm_key"].eq("")
    missing_either = missing_ri | missing_firm
    network = portfolio.loc[~missing_either].copy()
    duns_fallback = network["firm_identity_basis"].eq("valid DUNS fallback")
    name_fallback = network["firm_identity_basis"].eq(
        "unresolved surface-name fallback"
    )
    coverage_table = pd.DataFrame(
        [
            {
                "measure": "raw STTR source editions",
                "count": len(sttr_source),
                "denominator": "raw STTR source editions",
                "share": 1.0,
            },
            {
                "measure": "source-edition rows collapsed by canonical award key",
                "count": len(sttr_source) - len(portfolio),
                "denominator": "raw STTR source editions",
                "share": (len(sttr_source) - len(portfolio)) / len(sttr_source),
            },
            {
                "measure": "canonical STTR awards",
                "count": len(portfolio),
                "denominator": "canonical STTR awards",
                "share": 1.0,
            },
            {
                "measure": "award keys with 2+ source editions",
                "count": int(
                    edition_audit["observed_source_editions"].gt(1).sum()
                ),
                "denominator": "canonical STTR awards",
                "share": float(
                    edition_audit["observed_source_editions"].gt(1).mean()
                ),
            },
            {
                "measure": "award keys with variant source editions",
                "count": int(portfolio["source_edition_variants"].gt(1).sum()),
                "denominator": "canonical STTR awards",
                "share": float(
                    portfolio["source_edition_variants"].gt(1).mean()
                ),
            },
            {
                "measure": "award keys with network-field edition conflicts",
                "count": int(edition_audit["network_field_conflict"].sum()),
                "denominator": "canonical STTR awards",
                "share": float(edition_audit["network_field_conflict"].mean()),
            },
            {
                "measure": "missing RI name on chosen edition",
                "count": int(missing_ri.sum()),
                "denominator": "canonical STTR awards",
                "share": float(missing_ri.mean()),
            },
            {
                "measure": "RI name is a literal null token (within missing RI name)",
                "count": int(portfolio["ri_name_null_token"].sum()),
                "denominator": "canonical STTR awards",
                "share": float(portfolio["ri_name_null_token"].mean()),
            },
            {
                "measure": "missing firm endpoint on chosen edition",
                "count": int(missing_firm.sum()),
                "denominator": "canonical STTR awards",
                "share": float(missing_firm.mean()),
            },
            {
                "measure": "missing either endpoint on chosen edition",
                "count": int(missing_either.sum()),
                "denominator": "canonical STTR awards",
                "share": float(missing_either.mean()),
            },
            {
                "measure": "network-eligible canonical awards",
                "count": len(network),
                "denominator": "canonical STTR awards",
                "share": len(network) / len(portfolio),
            },
            {
                "measure": "eligible awards using valid DUNS fallback",
                "count": int(duns_fallback.sum()),
                "denominator": "network-eligible canonical awards",
                "share": float(duns_fallback.mean()),
            },
            {
                "measure": "eligible awards using unresolved surface-name fallback",
                "count": int(name_fallback.sum()),
                "denominator": "network-eligible canonical awards",
                "share": float(name_fallback.mean()),
            },
            {
                "measure": "eligible awards missing usable year",
                "count": int(network["observation_year"].isna().sum()),
                "denominator": "network-eligible canonical awards",
                "share": float(network["observation_year"].isna().mean()),
            },
        ]
    )
else:
    network = portfolio.iloc[0:0].copy()
    coverage_table = pd.DataFrame(
        [
            {
                "measure": "REFUSED — see load report",
                "count": pd.NA,
                "denominator": pd.NA,
                "share": pd.NA,
            }
        ]
    )
coverage_table


## Identity and source-key audit

This audit separates canonical award editions, identifier validity, and unresolved names. It cannot detect two different RI strings that are the same institution or one shared string that hides distinct institutions. A valid UEI carrying several company-name surfaces demonstrates why raw-name counts are not resolved firm counts; there is no equivalent RI identifier in this input.


In [ ]:
if analysis_ready:
    tracking_components = portfolio[["agency", "award_id", "phase"]].apply(
        lambda column: column.map(surface_key)
    )
    tracking_present = tracking_components.ne("").all(axis=1)
    tracking_phase_key = tracking_components.agg("|".join, axis=1)
    duplicate_tracking_phase_rows = int(
        tracking_phase_key.loc[tracking_present].duplicated(keep=False).sum()
    )
    ri_display_names = network["ri_name"].map(display_text)
    ri_surface_count = network["ri_key"].nunique()
    ri_display_count = ri_display_names.nunique()
    raw_uei_nonblank = portfolio["uei"].map(display_text).ne("")
    normalized_uei = portfolio["uei"].map(normalize_uei)
    uei_placeholder = normalized_uei.map(
        lambda value: placeholder_identifier(value if isinstance(value, str) else None)
    )
    raw_duns_nonblank = portfolio["duns"].map(display_text).ne("")
    normalized_duns = portfolio["duns"].map(normalize_duns)
    duns_placeholder = normalized_duns.map(
        lambda value: placeholder_identifier(value if isinstance(value, str) else None)
    )
    uei_company_variants = (
        portfolio.loc[portfolio["uei_key"].ne("") & portfolio["company_name_key"].ne("")]
        .groupby("uei_key")["company_name_key"]
        .nunique()
    )
    identity_audit = pd.DataFrame(
        [
            {
                "measure": "canonical award-key profile",
                "value": SBIR_AWARD_KEY_VERSION,
                "meaning": "identity computed on full raw source rows",
            },
            {
                "measure": "award keys with 2+ source editions",
                "value": int(edition_audit["observed_source_editions"].gt(1).sum()),
                "meaning": "collapsed by canonical edition selection",
            },
            {
                "measure": "award keys with RI-name edition conflicts",
                "value": int(edition_audit["ri_name_variants"].gt(1).sum()),
                "meaning": "chosen edition is deterministic; topology can be sensitive",
            },
            {
                "measure": "award keys with any network-field edition conflict",
                "value": int(edition_audit["network_field_conflict"].sum()),
                "meaning": "RI, company, valid UEI, or valid DUNS changed across editions",
            },
            {
                "measure": "award keys with title variants inside one grain",
                "value": int(edition_audit["title_variants"].gt(1).sum()),
                "meaning": "mutable titles do not create new canonical awards",
            },
            {
                "measure": "award keys with date-text variants inside one grain",
                "value": int(edition_audit["date_text_variants"].gt(1).sum()),
                "meaning": "canonical date formats can collapse to one award key",
            },
            {
                "measure": "duplicate agency+tracking+phase rows",
                "value": duplicate_tracking_phase_rows,
                "meaning": "the apparent award key is not assumed unique",
            },
            {
                "measure": "nonblank malformed UEI values on chosen editions",
                "value": int((raw_uei_nonblank & normalized_uei.isna()).sum()),
                "meaning": "rejected by canonical normalize_uei",
            },
            {
                "measure": "placeholder UEI values on chosen editions",
                "value": int(uei_placeholder.sum()),
                "meaning": "format-valid all-one-character values rejected after normalization",
            },
            {
                "measure": "nonblank malformed DUNS values on chosen editions",
                "value": int((raw_duns_nonblank & normalized_duns.isna()).sum()),
                "meaning": "rejected by canonical normalize_duns",
            },
            {
                "measure": "placeholder DUNS values on chosen editions",
                "value": int(duns_placeholder.sum()),
                "meaning": "format-valid all-one-character values rejected after normalization",
            },
            {
                "measure": "distinct trimmed RI display strings",
                "value": ri_display_count,
                "meaning": "before case-folding",
            },
            {
                "measure": "distinct RI surface-name nodes",
                "value": ri_surface_count,
                "meaning": "after Unicode/whitespace/case only",
            },
            {
                "measure": "valid UEIs with more than one firm-name surface",
                "value": int(uei_company_variants.gt(1).sum()),
                "meaning": "visible raw-name instability; not an alias-resolution result",
            },
        ]
    )
else:
    identity_audit = pd.DataFrame(columns=["measure", "value", "meaning"])
identity_audit


## Network overview and repeated dyads

A repeated dyad has at least two canonical awards for the same unresolved endpoint keys. This can reflect Phase I→II continuation, multiple projects, or repeated real-world relationships; it does not reveal contract form, institutional type, or relationship quality.


In [ ]:
def stable_display(series: pd.Series) -> str:
    values = series.map(display_text)
    values = values.loc[values.ne("")]
    if values.empty:
        return "(missing)"
    counts = values.value_counts()
    leaders = counts.loc[counts.eq(counts.max())].index.tolist()
    return sorted(leaders, key=str.casefold)[0]


def joined_labels(series: pd.Series) -> str:
    labels = {display_text(value) for value in series if display_text(value)}
    return "; ".join(sorted(labels, key=str.casefold))


def safe_share(numerator: int, denominator: int) -> float | None:
    return numerator / denominator if denominator else None


if analysis_ready and not network.empty:
    edges = (
        network.groupby(["firm_key", "ri_key"], sort=True, as_index=False)
        .agg(
            n_awards=("award_key", "size"),
            firm_label=("company_name", stable_display),
            ri_label=("ri_name", stable_display),
            first_observed_year=("observation_year", "min"),
            last_observed_year=("observation_year", "max"),
            n_agencies=("agency_key", "nunique"),
            phases=("phase", joined_labels),
        )
    )
    edges["repeat_dyad"] = edges["n_awards"].ge(2)
    repeated_award_count = int(edges.loc[edges["repeat_dyad"], "n_awards"].sum())
    observed_years = network["observation_year"].dropna().astype(int)
    year_span = (
        f"{observed_years.min()}–{observed_years.max()}"
        if not observed_years.empty
        else "unknown"
    )
    overview = pd.DataFrame(
        [
            {"measure": "canonical STTR awards", "value": len(portfolio)},
            {"measure": "network-eligible canonical awards", "value": len(network)},
            {"measure": "firm nodes", "value": network["firm_key"].nunique()},
            {"measure": "RI surface-name nodes", "value": network["ri_key"].nunique()},
            {"measure": "observed dyads", "value": len(edges)},
            {"measure": "dyads with 2+ awards", "value": int(edges["repeat_dyad"].sum())},
            {
                "measure": "share of dyads with 2+ awards",
                "value": safe_share(int(edges["repeat_dyad"].sum()), len(edges)),
            },
            {
                "measure": "share of eligible awards in repeated dyads",
                "value": safe_share(repeated_award_count, len(network)),
            },
            {"measure": "largest dyad award count", "value": int(edges["n_awards"].max())},
            {"measure": "observed year span", "value": year_span},
        ]
    )
else:
    edges = pd.DataFrame(
        columns=["firm_key", "ri_key", "n_awards", "repeat_dyad"]
    )
    overview = pd.DataFrame(
        [{"measure": "REFUSED — no eligible network awards", "value": pd.NA}]
    )
overview


In [ ]:
def count_band(
    value: int,
    bands: tuple[tuple[int | None, str], ...],
) -> str:
    for upper_bound, label in bands:
        if upper_bound is None or value <= upper_bound:
            return label
    raise AssertionError("count-band specification needs an open-ended final band")


EDGE_BANDS = (
    (1, "1 award"),
    (2, "2 awards"),
    (4, "3–4 awards"),
    (9, "5–9 awards"),
    (None, "10+ awards"),
)
EDGE_BAND_ORDER = [label for _upper_bound, label in EDGE_BANDS]
if not edges.empty:
    edge_weight_distribution = (
        edges["n_awards"]
        .map(lambda value: count_band(value, EDGE_BANDS))
        .value_counts()
        .reindex(EDGE_BAND_ORDER, fill_value=0)
        .rename_axis("awards_per_dyad")
        .rename("dyads")
        .reset_index()
    )
    edge_weight_distribution["share_of_dyads"] = (
        edge_weight_distribution["dyads"] / len(edges)
    )
else:
    edge_weight_distribution = pd.DataFrame(
        columns=["awards_per_dyad", "dyads", "share_of_dyads"]
    )
edge_weight_distribution


## First-observed and returning dyads by year

The annual unit is a dyad-year, not an award. Year uses the canonical pipeline's normalized `award_year`: source `Award Year` first, then parsed proposal-date year only when that field is absent. A dyad is `first_observed` in its earliest usable year in this export and `previously_observed` afterward. Several awards in the first year still count as one first-observed dyad. Awards with no usable year remain in overall topology but do not enter this table.


In [ ]:
known_year = network.loc[network["observation_year"].notna()].copy()
if analysis_ready and not known_year.empty:
    known_year["observation_year"] = known_year["observation_year"].astype(int)
    dyad_year = (
        known_year.groupby(
            ["firm_key", "ri_key", "observation_year"], as_index=False, sort=True
        )
        .agg(n_awards=("award_key", "size"))
    )
    dyad_year["first_observed_year"] = dyad_year.groupby(
        ["firm_key", "ri_key"]
    )["observation_year"].transform("min")
    dyad_year["status"] = dyad_year.apply(
        lambda row: (
            "first_observed"
            if row["observation_year"] == row["first_observed_year"]
            else "previously_observed"
        ),
        axis=1,
    )
    annual_dyads = (
        dyad_year.groupby("observation_year", as_index=False)
        .agg(
            active_dyads=("status", "size"),
            first_observed_dyads=("status", lambda s: int(s.eq("first_observed").sum())),
            previously_observed_dyads=(
                "status",
                lambda s: int(s.eq("previously_observed").sum()),
            ),
            network_awards=("n_awards", "sum"),
        )
        .sort_values("observation_year")
    )
    annual_dyads["returning_dyad_share"] = (
        annual_dyads["previously_observed_dyads"] / annual_dyads["active_dyads"]
    )
else:
    dyad_year = pd.DataFrame(
        columns=["firm_key", "ri_key", "observation_year", "n_awards", "status"]
    )
    annual_dyads = pd.DataFrame(
        columns=[
            "observation_year",
            "active_dyads",
            "first_observed_dyads",
            "previously_observed_dyads",
            "network_awards",
            "returning_dyad_share",
        ]
    )
annual_dyads


## Award concentration and partner reuse

Concentration uses eligible canonical-award shares attached to unresolved RI surface names and dyads. HHI is the sum of squared award shares on a 0–1 scale. It is a compact portfolio-distribution statistic only. The ranked RI table must not be read as a ranking of resolved institutions.


In [ ]:
def concentration_rows(counts: pd.Series, population: str) -> list[dict[str, object]]:
    ordered = counts.sort_values(ascending=False)
    total = int(ordered.sum())
    shares = ordered / total if total else ordered.astype(float)
    rows: list[dict[str, object]] = []
    for top_n in (1, 5, 10):
        rows.append(
            {
                "population": population,
                "measure": f"top {top_n} award share",
                "value": float(shares.head(top_n).sum()) if total else None,
            }
        )
    rows.append(
        {
            "population": population,
            "measure": "award-share HHI (0–1)",
            "value": float(shares.pow(2).sum()) if total else None,
        }
    )
    return rows


if analysis_ready and not network.empty:
    ri_ranked = (
        network.groupby("ri_key", as_index=False)
        .agg(
            ri_surface_label=("ri_name", stable_display),
            n_awards=("award_key", "size"),
            n_firms=("firm_key", "nunique"),
        )
        .sort_values(["n_awards", "ri_surface_label"], ascending=[False, True])
    )
    ri_ranked["award_share"] = ri_ranked["n_awards"] / len(network)
    dyad_counts = edges.set_index(["firm_key", "ri_key"])["n_awards"]
    concentration_table = pd.DataFrame(
        concentration_rows(ri_ranked.set_index("ri_key")["n_awards"], "RI surface-name nodes")
        + concentration_rows(dyad_counts, "firm–RI surface-name dyads")
    )
else:
    ri_ranked = pd.DataFrame(
        columns=["ri_key", "ri_surface_label", "n_awards", "n_firms", "award_share"]
    )
    concentration_table = pd.DataFrame(columns=["population", "measure", "value"])
concentration_table


In [ ]:
# Public organization strings only; no RI contact fields were loaded.
ri_ranked.head(25)


In [ ]:
NODE_BANDS = (
    (1, "1"),
    (2, "2"),
    (4, "3–4"),
    (9, "5–9"),
    (None, "10+"),
)
NODE_BAND_ORDER = [label for _upper_bound, label in NODE_BANDS]
if not edges.empty:
    firm_reuse = (
        edges.groupby("firm_key", as_index=False)
        .agg(
            n_ri_partners=("ri_key", "nunique"),
            n_awards=("n_awards", "sum"),
            repeated_dyads=("repeat_dyad", "sum"),
        )
    )
    ri_reuse = (
        edges.groupby("ri_key", as_index=False)
        .agg(
            n_firms=("firm_key", "nunique"),
            n_awards=("n_awards", "sum"),
            repeated_dyads=("repeat_dyad", "sum"),
        )
    )
    reuse_summary = pd.DataFrame(
        [
            {
                "measure": "firms observed with 2+ RI surface partners",
                "nodes": int(firm_reuse["n_ri_partners"].ge(2).sum()),
                "share": float(firm_reuse["n_ri_partners"].ge(2).mean()),
            },
            {
                "measure": "firms with at least one repeated dyad",
                "nodes": int(firm_reuse["repeated_dyads"].ge(1).sum()),
                "share": float(firm_reuse["repeated_dyads"].ge(1).mean()),
            },
            {
                "measure": "RI surface nodes observed with 2+ firms",
                "nodes": int(ri_reuse["n_firms"].ge(2).sum()),
                "share": float(ri_reuse["n_firms"].ge(2).mean()),
            },
        ]
    )
    firm_bands = (
        firm_reuse["n_ri_partners"]
        .map(lambda value: count_band(value, NODE_BANDS))
        .value_counts()
        .reindex(NODE_BAND_ORDER, fill_value=0)
        .rename("nodes")
        .rename_axis("partner_count_band")
        .reset_index()
        .assign(node_population="firm nodes")
    )
    ri_bands = (
        ri_reuse["n_firms"]
        .map(lambda value: count_band(value, NODE_BANDS))
        .value_counts()
        .reindex(NODE_BAND_ORDER, fill_value=0)
        .rename("nodes")
        .rename_axis("partner_count_band")
        .reset_index()
        .assign(node_population="RI surface-name nodes")
    )
    partner_reuse_bands = pd.concat([firm_bands, ri_bands], ignore_index=True)
    partner_reuse_bands["share_within_population"] = partner_reuse_bands.groupby(
        "node_population"
    )["nodes"].transform(lambda s: s / s.sum())
else:
    firm_reuse = pd.DataFrame()
    ri_reuse = pd.DataFrame()
    reuse_summary = pd.DataFrame(columns=["measure", "nodes", "share"])
    partner_reuse_bands = pd.DataFrame(
        columns=["partner_count_band", "nodes", "node_population", "share_within_population"]
    )
reuse_summary


In [ ]:
partner_reuse_bands


## Agency and agency-year variation

Agency tables retain RI-name coverage beside topology. Repeat share is calculated within each agency; top-RI share and HHI remain unresolved surface-name award statistics. In the agency-year table, `globally_first_observed_dyads` and `globally_previously_observed_dyads` use the dyad's history across all agencies, not within-agency history. Differences can reflect agency data practices, program scale, mission, phase mix, years covered, and naming conventions. They are not agency effects.


In [ ]:
if analysis_ready:
    agency_rows: list[dict[str, object]] = []
    for agency_key, base in portfolio.groupby("agency_key", sort=True):
        agency_network = network.loc[network["agency_key"].eq(agency_key)]
        agency_label = stable_display(base["agency"])
        if agency_network.empty:
            agency_edges = pd.Series(dtype=int)
            ri_counts = pd.Series(dtype=int)
        else:
            agency_edges = agency_network.groupby(["firm_key", "ri_key"]).size()
            ri_counts = agency_network.groupby("ri_key").size().sort_values(ascending=False)
        ri_shares = ri_counts / ri_counts.sum() if len(ri_counts) else ri_counts.astype(float)
        agency_rows.append(
            {
                "agency": agency_label,
                "canonical_awards": len(base),
                "ri_name_coverage": float(base["ri_key"].ne("").mean()),
                "eligible_network_awards": len(agency_network),
                "firm_nodes": agency_network["firm_key"].nunique(),
                "ri_surface_nodes": agency_network["ri_key"].nunique(),
                "dyads": len(agency_edges),
                "repeat_dyad_share_within_agency": (
                    float(agency_edges.ge(2).mean()) if len(agency_edges) else None
                ),
                "top_ri_award_share": (float(ri_shares.head(1).sum()) if len(ri_shares) else None),
                "ri_award_share_hhi": (float(ri_shares.pow(2).sum()) if len(ri_shares) else None),
            }
        )
    agency_summary = pd.DataFrame(agency_rows).sort_values(
        ["canonical_awards", "agency"], ascending=[False, True]
    )
else:
    agency_summary = pd.DataFrame(
        columns=[
            "agency",
            "canonical_awards",
            "ri_name_coverage",
            "eligible_network_awards",
            "firm_nodes",
            "ri_surface_nodes",
            "dyads",
            "repeat_dyad_share_within_agency",
            "top_ri_award_share",
            "ri_award_share_hhi",
        ]
    )
agency_summary


In [ ]:
if analysis_ready and not portfolio["observation_year"].dropna().empty:
    agency_year_rows: list[dict[str, object]] = []
    dated_portfolio = portfolio.loc[portfolio["observation_year"].notna()].copy()
    dated_portfolio["observation_year"] = dated_portfolio["observation_year"].astype(int)
    first_year_lookup = (
        dyad_year.groupby(["firm_key", "ri_key"])["observation_year"].min()
        if not dyad_year.empty
        else pd.Series(dtype=int)
    )
    for (_agency_key, year), base in dated_portfolio.groupby(
        ["agency_key", "observation_year"], sort=True
    ):
        agency_label = stable_display(base["agency"])
        cell_network = base.loc[base["network_eligible"]].copy()
        cell_edges = (
            cell_network.groupby(["firm_key", "ri_key"]).size().rename("n_awards")
            if not cell_network.empty
            else pd.Series(dtype=int)
        )
        globally_first_count = 0
        globally_previous_count = 0
        for dyad in cell_edges.index:
            first_year = first_year_lookup.get(dyad)
            if first_year == year:
                globally_first_count += 1
            elif pd.notna(first_year) and first_year < year:
                globally_previous_count += 1
        agency_year_rows.append(
            {
                "agency": agency_label,
                "observation_year": int(year),
                "canonical_awards": len(base),
                "ri_name_coverage": float(base["ri_key"].ne("").mean()),
                "eligible_network_awards": len(cell_network),
                "active_dyads": len(cell_edges),
                "globally_first_observed_dyads": globally_first_count,
                "globally_previously_observed_dyads": globally_previous_count,
                "firm_nodes": cell_network["firm_key"].nunique(),
                "ri_surface_nodes": cell_network["ri_key"].nunique(),
            }
        )
    agency_year_all = pd.DataFrame(agency_year_rows).sort_values(
        ["observation_year", "agency"]
    )
    agency_year_display = agency_year_all.loc[
        agency_year_all["canonical_awards"].ge(MIN_AGENCY_YEAR_AWARDS)
    ].copy()
else:
    agency_year_all = pd.DataFrame()
    agency_year_display = pd.DataFrame(
        columns=["agency", "observation_year", "canonical_awards"]
    )
agency_year_display


The agency-year display threshold suppresses only small cells from the rendered table; `agency_year_all` retains them. It is not a statistical threshold or an exclusion from any other result.


## Deterministic integrity checks

These checks enforce the declared grains and accounting identities. Adversarial probes also verify that mutable title/date-format editions collapse and malformed or placeholder identifiers are rejected. They do not validate entity identity or source completeness.


In [ ]:
edition_probe_base = {
    "source_row": 2,
    "Agency Tracking Number": "TRACK-1",
    "Contract": "CONTRACT-1",
    "Company": "Probe Company",
    "Agency": "DEFENSE",
    "Branch": "NAVY",
    "Phase": "Phase II",
    "Program": "STTR",
    "Proposal Award Date": "2026-06-12",
    "Award Year": "2027",
    "Solicitation Number": "SOL-1",
    "UEI": "ABC123DEF456",
    "Duns": "123456789",
    "Award Title": "Original mutable title",
    "Award Amount": "100",
    "Contract End Date": "2027-01-01",
}
edition_probe_changed = {
    **edition_probe_base,
    "source_row": 3,
    "Proposal Award Date": "06/12/2026",
    "Award Title": "Revised mutable title",
    "Award Amount": "200",
    "Contract End Date": "2027-06-01",
}
assert sbir_award_grain_key(edition_probe_base) == sbir_award_grain_key(
    edition_probe_changed
)
probe_collapsed = normalize_sbir_awards(
    pd.DataFrame([edition_probe_base, edition_probe_changed])
)
assert len(probe_collapsed) == 1
assert int(probe_collapsed.iloc[0]["source_edition_count"]) == 2
assert int(probe_collapsed.iloc[0]["source_edition_variants"]) == 2
assert int(probe_collapsed.iloc[0]["award_year"]) == 2027
assert surface_key("N/A") == "n/a"
assert is_sttr(" sttr ")
assert valid_uei("abc-123-def-456") == "ABC123DEF456"
assert valid_uei("short") is None
assert valid_uei("000000000000") is None
assert valid_duns("12-345-6789") == "123456789"
assert valid_duns("000000000") is None
assert firm_identity("ABC123DEF456", "123456789", "probe") == (
    "UEI:ABC123DEF456",
    "valid UEI",
)
assert firm_identity("", "123456789", "probe") == (
    "DUNS:123456789",
    "valid DUNS fallback",
)

if analysis_ready:
    assert sttr_source["award_key_version"].eq(SBIR_AWARD_KEY_VERSION).all()
    assert portfolio["award_key"].is_unique
    assert portfolio["observation_year"].equals(
        pd.to_numeric(portfolio["canonical_award_year"]).astype("Int64")
    )
    assert (
        len(sttr_source) - len(portfolio)
        == load_report["edition_rows_collapsed"]
    )
    expected_editions = sttr_source.groupby("award_key").size().sort_index()
    observed_editions = (
        portfolio.set_index("award_key")["source_edition_count"]
        .astype(int)
        .sort_index()
    )
    pd.testing.assert_series_equal(
        observed_editions, expected_editions, check_names=False
    )
    assert int(portfolio["network_eligible"].sum()) == len(network)
    assert int(edges["n_awards"].sum()) == len(network)
    if not annual_dyads.empty:
        assert (
            annual_dyads["first_observed_dyads"]
            + annual_dyads["previously_observed_dyads"]
            == annual_dyads["active_dyads"]
        ).all()
    if not agency_summary.empty:
        assert int(agency_summary["canonical_awards"].sum()) == len(portfolio)
    if not agency_year_all.empty:
        assert (
            agency_year_all["globally_first_observed_dyads"]
            + agency_year_all["globally_previously_observed_dyads"]
            == agency_year_all["active_dyads"]
        ).all()
        assert int(agency_year_all["canonical_awards"].sum()) == int(
            portfolio["observation_year"].notna().sum()
        )
    print("PASS — award grain, edges, year status, and agency accounting hold.")
else:
    print(f"SKIPPED — analysis refused: {load_report['reason']}")


## Findings and refusals

| Permitted descriptive reading | Table | Required caveat |
|---|---|---|
| How many observed surface-name dyads have one versus several canonical awards | `overview`, `edge_weight_distribution` | Repetition can be a phase continuation or another award; endpoint identity remains unresolved |
| How the first-observed/previously-observed mix changes across usable award years | `annual_dyads` | First-observed is snapshot entry, not relationship formation |
| How canonical awards are distributed across RI surface names and dyads | `concentration_table`, `ri_ranked` | Surface names can split or merge institutions; concentration is not market power or fairness |
| How many firm nodes use several RI surface names and how many RI names appear with several firms | `reuse_summary`, `partner_reuse_bands` | This is observed partner reuse under unresolved keys, not durable institutional identity |
| How coverage and descriptive topology vary by agency and year | `agency_summary`, `agency_year_all` | Agency-year first/previous status uses each dyad's global observed history; differences are not agency effects |
| What share of canonical funded awards lacks a usable RI or firm endpoint | `coverage_table` | Missing RI text is unknown, never “no partner” |

**Refusals:** This notebook does not identify institution types; classify spinouts or subcontracts; infer relationship quality; compare funded firms with applicants; measure access or equity; estimate causal effects; or claim a complete federal network. Any such use requires a different question, additional data, entity-resolution review, and explicit promotion from the exploratory tier.

If the input cell says `REFUSED`, there is no numerical finding. Set `SBIR_AWARDS_PATH` to a raw SBIR.gov CSV. The preferred pin is its canonical sibling metadata sidecar. When that sidecar is absent, also set `SBIR_AWARDS_SHA256` to the observed hash and `SBIR_AWARDS_AS_OF` to a valid `YYYY-MM-DD` source date, then rerun all cells.


## Promotion checklist

- [x] Question and decision are explicit.
- [x] Input selection, as-of field, population, grains, keys, exclusions, assumptions, and seed are recorded.
- [x] Missingness stays in the canonical funded-award denominator and is separate from network topology.
- [x] Identity limitations and prohibited interpretations are prominent.
- [x] The destination tier is explicit (`exploratory`, non-citable).
- [ ] RI and fallback firm names are resolved and reviewed against a governed identity artifact.
- [x] Source identity comes from a fully verified canonical sidecar or a validated hash/date fallback.
- [ ] Recurring calculations have one canonical implementation and verifier.
- [ ] Citable work satisfies all four `evidence` contract requirements.
- [x] Outputs and execution counts are cleared before commit.

Until promotion is complete, this notebook and every number it displays remain exploratory and non-citable.
